# Analyzing the Impact of Recipe Characteristics on User Ratings

This project aims to investigate how recipe characteristics such as preparation time, number of ingredients, and nutritional values influence user ratings on online platforms. Understanding these relationships can provide insights into user preferences and recipe popularity.

Data Sources:
- Food.com Recipes Dataset
- External Nutrition Dataset

In this notebook, we merge the datasets and evaluate how recipe features relate to ratings.

Libraries used: `pandas`, `seaborn`, `scipy`

In [ ]:
import ast
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', context='notebook')

## 1. Data Loading and Merging

The cell below looks for files in the `data/` folder. If needed, you can directly replace them with your own file names.

In [ ]:
# 1) Load datasets
data_dir = Path('data')

# Expected files (change if needed)
recipes_path = data_dir / 'RAW_recipes.csv'
interactions_path = data_dir / 'RAW_interactions.csv'
nutrition_path = data_dir / 'nutrition.csv'

recipes = pd.read_csv(recipes_path)
interactions = pd.read_csv(interactions_path)
nutrition = pd.read_csv(nutrition_path)

print('recipes shape:', recipes.shape)
print('interactions shape:', interactions.shape)
print('nutrition shape:', nutrition.shape)

In [ ]:
# 1) Prepare common columns for merging
# In Food.com data, recipe id is usually 'id' in recipes and 'recipe_id' in interactions.
if 'id' in recipes.columns and 'recipe_id' in interactions.columns:
    recipes = recipes.rename(columns={'id': 'recipe_id'})

# Recipe-level average rating and rating count
ratings_per_recipe = (
    interactions.groupby('recipe_id', as_index=False)['rating']
.agg(avg_rating='mean', rating_count='count')
)

# Merge recipes with rating summary
df = recipes.merge(ratings_per_recipe, on='recipe_id', how='left')

# Merge with nutrition dataset
# First, align common key names.
nutrition = nutrition.copy()
if 'id' in nutrition.columns and 'recipe_id' not in nutrition.columns:
    nutrition = nutrition.rename(columns={'id': 'recipe_id'})

if 'recipe_id' in nutrition.columns and 'recipe_id' in df.columns:
    df = df.merge(nutrition, on='recipe_id', how='left', suffixes=('', '_nut'))
elif 'name' in nutrition.columns and 'name' in df.columns:
    df = df.merge(nutrition, on='name', how='left', suffixes=('', '_nut'))
else:
    raise ValueError('No merge key found in nutrition dataset: expected recipe_id or name.')

# Ready for inspection

In [ ]:
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

In [ ]:
df.isna().sum().sort_values(ascending=False)

## 2. Data Cleaning

After merging the datasets, the data is cleaned by handling missing values and preparing variables required for analysis.

In this section:
- We handle missing values
- We create the `ingredient_count` feature

In [ ]:
# 2) Missing values
key_cols = ['avg_rating', 'minutes']
for col in key_cols:
    if col not in df.columns:
        raise ValueError(f'Expected column not found: {col}')

# Detect nutrition columns
nutrition_cols = [c for c in ['calories', 'fat', 'protein', 'sugar', 'sodium'] if c in df.columns]

# Create ingredient_count: ingredients column may be stored as a stringified list
def count_ingredients(x):
    if pd.isna(x):
        return np.nan
    if isinstance(x, list):
        return len(x)
    if isinstance(x, str):
        x = x.strip()
        try:
            parsed = ast.literal_eval(x)
            if isinstance(parsed, list):
                return len(parsed)
        except (ValueError, SyntaxError):
            # In some datasets, ingredients can be a simple comma-separated string
            if ',' in x:
                return len([p for p in x.split(',') if p.strip()])
        return np.nan
    return np.nan

if 'ingredients' in df.columns:
    df['ingredient_count'] = df['ingredients'].apply(count_ingredients)
else:
    raise ValueError('ingredients column not found. ingredient_count could not be created.')

# Create an ingredient complexity group for later boxplot analysis
ingredient_median = df['ingredient_count'].median()
df['ingredient_group'] = np.where(df['ingredient_count'] <= ingredient_median, 'fewer_ingredients', 'more_ingredients')

# Drop missing values in analysis columns
analysis_cols = ['avg_rating', 'minutes', 'ingredient_count', 'ingredient_group'] + nutrition_cols
df_clean = df.dropna(subset=['avg_rating', 'minutes', 'ingredient_count']).copy()

# Fill missing nutrition values with column medians (optional but practical)
for col in nutrition_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

print('Cleaned dataset shape:', df_clean.shape)
display(df_clean[analysis_cols].head())

## 3. Exploratory Data Analysis

Exploratory data analysis helps reveal patterns, distributions, and relationships between variables before applying statistical tests.

In [ ]:
# 3a) Rating distribution
plt.figure(figsize=(7, 4))
sns.histplot(df_clean['avg_rating'], bins=20, kde=True)
plt.title('Average Rating Distribution')
plt.xlabel('Average Rating')
plt.ylabel('Count')
plt.show()

print('Interpretation: The distribution of ratings shows that most recipes receive high ratings, indicating a possible bias toward positive feedback.')
print('Interpretation: Because the distribution is concentrated near higher values, small differences between features may be harder to detect.')

In [ ]:
# 3b) Preparation time vs rating
plt.figure(figsize=(7, 4))
sns.scatterplot(data=df_clean, x='minutes', y='avg_rating', alpha=0.35)
plt.title('Preparation Time vs Average Rating')
plt.xlabel('Preparation Time (minutes)')
plt.ylabel('Average Rating')
plt.show()

print('Interpretation: There is a weak negative relationship between preparation time and rating.')
print('Interpretation: The relationship appears weak and scattered, indicating that preparation time alone is not a strong predictor of ratings.')

if 'calories' in df_clean.columns:
    plt.figure(figsize=(7, 4))
    sns.scatterplot(data=df_clean, x='calories', y='avg_rating', alpha=0.35)
    plt.title('Calories vs Average Rating')
    plt.xlabel('Calories')
    plt.ylabel('Average Rating')
    plt.show()

    print('Interpretation: No clear linear pattern is visible between calories and ratings, and the points largely overlap.')
    print('Interpretation: This suggests calorie level does not strongly affect ratings on its own.')

plt.figure(figsize=(7, 4))
sns.scatterplot(data=df_clean, x='ingredient_count', y='avg_rating', alpha=0.35)
plt.title('Ingredient Count vs Average Rating')
plt.xlabel('Ingredient Count')
plt.ylabel('Average Rating')
plt.show()

print('Interpretation: Ingredient count shows a slight negative relationship with ratings.')
print('Interpretation: The effect appears modest and scattered, so ingredient count alone is not a strong predictor of ratings.')

### Bonus EDA: Correlation Heatmap and Ingredient Group Boxplot

In [ ]:
# Correlation heatmap for numeric variables
numeric_cols = ['avg_rating', 'minutes', 'ingredient_count'] + [c for c in ['calories', 'fat', 'protein', 'sugar', 'sodium'] if c in df_clean.columns]
corr_matrix = df_clean[numeric_cols].corr(numeric_only=True)

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt='.2f', square=True)
plt.title('Correlation Heatmap of Key Features')
plt.show()

print('Interpretation: Correlation values close to 0 indicate weak linear relationships with rating.')
print('Interpretation: This helps identify which variables are potentially useful for deeper modeling.')

# Boxplot: ingredient complexity groups vs ratings
plt.figure(figsize=(7, 4))
sns.boxplot(x='ingredient_group', y='avg_rating', data=df_clean)
plt.title('Ratings by Ingredient Complexity Group')
plt.xlabel('Ingredient Group')
plt.ylabel('Average Rating')
plt.show()

print('Interpretation: If the median line is higher for fewer_ingredients, simpler recipes tend to receive slightly better ratings.')
print('Interpretation: Overlapping boxes indicate the difference may be small and should be validated statistically.')

## 4. Hypothesis Testing

Hypothesis testing is used to determine whether observed differences in ratings are statistically significant or due to random variation.

We test the following hypotheses:

- H1: Recipes with shorter preparation times receive higher ratings.
- H2: Recipes with lower calorie values receive higher ratings.
- H3: Recipes with fewer ingredients receive higher ratings.

Method: We split each feature into two groups by median and apply an independent-samples t-test.

In [ ]:
def run_ttest_by_median(df_in, feature, target='avg_rating', alpha=0.05):
    temp = df_in[[feature, target]].dropna().copy()
    threshold = temp[feature].median()

    low_group = temp[temp[feature] <= threshold][target]
    high_group = temp[temp[feature] > threshold][target]

    t_stat, p_two_sided = stats.ttest_ind(low_group, high_group, equal_var=False, nan_policy='omit')

    # One-tailed p-value for directional hypothesis
    mean_low = low_group.mean()
    mean_high = high_group.mean()
    if mean_low > mean_high:
        p_one_sided = p_two_sided / 2
    else:
        p_one_sided = 1 - (p_two_sided / 2)

    reject_h0 = p_one_sided < alpha

    return {
        'feature': feature,
        'median_threshold': threshold,
        'mean_low_feature_group': mean_low,
        'mean_high_feature_group': mean_high,
        't_stat': t_stat,
        'p_two_sided': p_two_sided,
        'p_one_sided': p_one_sided,
        'reject_h0': reject_h0
    }

results = []
results.append(run_ttest_by_median(df_clean, 'minutes'))

if 'calories' in df_clean.columns:
    results.append(run_ttest_by_median(df_clean, 'calories'))
else:
    print('H2 test skipped because calories column was not found.')

results.append(run_ttest_by_median(df_clean, 'ingredient_count'))

results_df = pd.DataFrame(results)
display(results_df)

def print_hypothesis_result(label, feature_key, interpretation_text):
    row = results_df[results_df['feature'] == feature_key].iloc[0]
    pval = row['p_one_sided']
    low_mean = row['mean_low_feature_group']
    high_mean = row['mean_high_feature_group']
    direction_ok = low_mean > high_mean

    reject_h0 = (pval < 0.05 and direction_ok)

    print(f'\n{label} Result:')
    print(f'P-value: {pval:.4f}')
    print('Decision:', 'Reject H0' if reject_h0 else 'Fail to reject H0')
    if reject_h0:
        print('Since the p-value is less than 0.05 and the group means are in the expected direction, we reject the null hypothesis.')
    else:
        print('Since the p-value is greater than 0.05 or the observed direction does not match the hypothesis, we fail to reject the null hypothesis.')
    print(f'Interpretation: {interpretation_text} {'does' if reject_h0 else 'does not'} significantly affect ratings.')
    if not reject_h0:
        print('No statistically significant difference was found between the low and high groups for this feature.')

print_hypothesis_result('H1', 'minutes', 'Shorter preparation time')

if 'calories' in results_df['feature'].values:
    print_hypothesis_result('H2', 'calories', 'Lower calorie level')
else:
    print('\nH2 Result: Could not be evaluated because calories data is unavailable.')

print_hypothesis_result('H3', 'ingredient_count', 'Having fewer ingredients')

## 5. Results Summary

Key Findings:
- There is a weak negative relationship between preparation time and ratings.
- No statistically significant difference was found across calorie-based groups.
- Ingredient count shows a slight negative effect, but the relationship is modest.
- The correlation heatmap also indicates mostly weak linear associations with ratings.

Overall, the analysis suggests that basic recipe features such as preparation time, calorie content, and ingredient count have limited influence on user ratings. This indicates that other factors may play a more important role.
These findings suggest that simple measurable features are not sufficient to explain user ratings, and more complex factors may play a significant role.
These results highlight that observable numerical features alone are not sufficient to explain user preferences.

Note: A natural next step is to test these effects with multivariable regression.

## Limitations

- Ratings are user-generated and may contain positivity bias.
- The analysis tests direct pairwise relationships; interaction effects are not modeled here.
- Important unobserved factors (taste, cuisine familiarity, photo quality, reviewer behavior) are not directly captured.
- Results are correlational and should not be interpreted as causal effects.